PCA, t-sne, isolation forest - 3가지의 기술은 비선형 데이터를 다룰 때 중요한 기술이다.  비선형의 특징을 잡아내는데 사용된다.

- PCA : 비선형에는 약하지만 징검다리로 필수, 고속 압축기
- t-sne : 고차원 형태의 것을 2차원으로 만들어서 표현한다., 비선형 시각화 마술사
- isolation forest : 노이즈를 찾아낸다.
고차원형태 비선형이 아닐때 기술을 사용해서 연산이 가능하게끔, 편안하게끔하는 것
학습시키기위해서 2차원형태로 바꿔서 학습.

In [ ]:
#PCA
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names

print(f'원본 데이터 형태(shape):{X.shape} 즉 {X.shape[1]}차원이다.')
print("원본 특성 이름(열/특징(feature)/차원)",feature_names)

# PCA 실행전에 반드시 표준 스케일링을 한다.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA 모델 생성 및 학습 / 4차원을 2차원으로 압축
pca = PCA(n_components=2)  # 2차원으로 축소
X_pca = pca.fit_transform(X_scaled)

print(f'축소된 데이터 형태(shape):{X_pca.shape}')

print(f'설명된 분산 비율:{pca.explained_variance_ratio_}')
print(f"보존된 총 정보량:{sum(pca.explained_variance_ratio_) * 100 :.2f}")

df_pca = pd.DataFrame(data=X_pca, columns=['PC1','PC2'])
print(df_pca.head())
df_pca['target'] = y

df_pca['Target_Name'] = df_pca['target'].map({
    0:iris.target_names[0], #'setosa',
    1:iris.target_names[1], #'versicolor',
    2:iris.target_names[2]  # virginica'
})

plt.figure(figsize=(10,6))
sns.scatterplot(
    data=df_pca,
    x='PC1',
    y='PC2',
    hue='Target_Name', #색조
    palette='Set1',
    s=80,
    alpha=0.9)
plt.title('PCA Dimensionality Reduction(Iris Dataset: 4D -> 2D)', fontsize = 16, fontweight = 'bold')
plt.xlabel('PC1', fontsize = 11)
plt.ylabel('PC2', fontsize = 11)
plt.legend(title= 'Iris Variety', loc='upper right')
plt.grid(True, linestyle = '--', alpha = 0.5)
plt.show()


isolation Forest: 이상 탐지

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
#정상 데이터
np.random.seed(42)
X_normal = np.random.normal(loc=0, scale=1, size=(200,2))

#이상치 데이터
X_outliers = np.random.uniform(low=-10, high=10, size=(20,2))

# 정상과 이상치 결합(220)
X = np.vstack((X_normal, X_outliers))

print(f"전체 데이터 형태:{X.shape}")
print(f"정상 데이터 형태:{X_normal.shape}")
print(f"이상치 데이터 형태: {X_outliers.shape}")

iso_forest = IsolationForest(contamination=0.1, random_state=42)
# 모델 예측 수행
preds = iso_forest.fit_predict(X)

num_outliers = np.sum(preds == -1)
print(f"이상치 개수: {num_outliers}개 감지됨.")

plt.figure(figsize=(10,6))

colors = ['crimson' if p == -1 else 'dodgerblue' for p in preds]
plt.scatter(
    X[:, 0],
    X[:, 1],
    c=colors,
    alpha=0.85,
    s= 70,
    edgecolors='k'
)
plt.title("Isolation Forest Anomaly Detection", fontsize= 13, fontweight='bold')
plt.xlabel("Feature 1", fontsize = 11)
plt.ylabel("Feature 2", fontsize = 11)
plt.grid(True, linestyle= '--',alpha=0.5)
plt.show()

PCA vs t-SNE 차이
- PCA : 전역적 구조(데이터 전체의 분산과 거리를 최대한 보존). 차원 축소 후 머신러닝 모델의 입력값으로 사용하기 좋음.
- T-SNE : 국소적 구조(내 주변 가까운 이웃과의 관계를 극도로 보존). 오직 시각화 목적에 특화됨

In [ ]:
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

digits = load_digits()
X = digits.data
y = digits.target

print(f"고차원 원본 데이터 형태: {X.shape}")
print("특성 개수:",X.shape[1],"차원")

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X)

print(f"t-SNE 적용 후 데이터 형태: {X_tsne.shape}")
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=X_tsne[:, 0],
    y=X_tsne[:, 1],
    hue=y,
    s=60,
    palette='tab10',
    alpha=0.85,
    legend='full'
)
plt.title('t-sne(64D -> 2D)',fontsize=14,fontweight='bold')
plt.xlabel('t-SNE 1st dim',fontsize=14)
plt.ylabel('t-SNE 2nd dim',fontsize=14)
plt.legend(title='digites (0-9)',bbox_to_anchor=(1.02,1), loc='upper left')
plt.grid(True, linestyle='--', alpha = 0.5)
plt.tight_layout()
plt.show()

텍스트 군집화

TF-IDF
- TF(단어 빈도) : 특정 문서안에서 그 단어가 몇 번 나타났는가?
- IDF(역문서 빈도) : 그 단어가 전체 문서들 중에서 얼마나 희귀한가?
1. 문장들을 모아 TfidfVectorizer에 통과시켜 문서-단어 행렬을 얻는다.
2. 이 행렬은 결국 수많은 숫자로 이루어진 특성들이므로, 그대로 Kmeans 알고리즘에 집어 넣어 거리를 계산하며 끝

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

documents = [
    "Machine learning is a branch of artificial intelligence.",
    "Deep machine learning uses artificial intelligence and neural networks.",
    "Artificial intelligence and machine learning are transforming industries.",
    "Cooking recipes often require flour, sugar, and butter.",
    "Baking a cake is a fun and delicious activity.",
    "Delicious chocolate cake recipe with sweet frosting."
]
print(f"총 문서 수: {len(documents)}개")

vectorizer = TfidfVectorizer(stop_words='english')

X_tfidf = vectorizer.fit_transform(documents)

print(f"변환된 TF-IDF 행렬 형태(문서수,단어수)", X_tfidf.shape)
kmeans = KMeans(n_clusters=2, random_state=42, n_init= 10)
kmeans.fit(X_tfidf)

print("=====텍스트 비지도 군집화 출력결과=====")
for doc,cluster_id in zip(documents, kmeans.labels_):
  print(f"[Cluster {cluster_id}]-> {doc}")

PyTorch 텐서플로
0차원 : 스칼라
1차원 : 벡터
2차원 : 행렬
그이후부터는 텐서

In [ ]:
import torch

print("현재 사용중인 PyTorch 버전:", torch.__version__)
print("GPU 사용 가능 여부", torch.cuda.is_available())

#기본 텐서 생
tensor_2d = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print("기본 2차원 텐서")
print(tensor_2d)
print("텐서 형태(shape)", tensor_2d.shape)
print("텐서 차원(dimension)", tensor_2d.ndim)
print("텐서 데이터타입(dtype)", tensor_2d.dtype)
print("연산이 수행되는장소(Device)", tensor_2d.device)

- 역전파: 예측 오류를 바탕으로 스스로 학습할 수 있게 만드는 핵심 알고리즘 (뒤에서 앞으로)
- 순전파 : 문제를 푸는 과정(앞에서 뒤로 입력데이터를 넣고 푼다.)  
- requires_grad=True와 .backward()를 결합하면 복잡한 수식의 미분값을 코드 한 줄로 자동 계산할 수 있다.

In [ ]:
# 리스트를 텐서로 변환
x = torch.tensor([3.0,4.0], requires_grad=True)

print("입력 텐서 x: ", x)
y = 2 * (x**2) + 5*x +1
print("결과 텐서 y:", y)

loss = y.sum()               #y는 벡터이므로 더해서  스칼라로 만든다. 그 후에 역전파를 수행한다.
print("scalar loss:", loss)
loss.backward()

print("자동 미분 결과 (dy/dx)")
print(x.grad)

- 미니배치(Mini-batch)
- Dataset
- DataLoader
- PyTorch에서 제공하는 Dataset은 추상 클래스이다. 반드시 오버라이딩을 해야하는 필수 메소드가 있다.
  - init(self): 초기화
  - len(self): 전체 길이반환
  - getitem(self.idx):


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. torch.utils.dataset을 상속받는 커스텀 데이터셋 클래스 정의
class MyCustomDataset(Dataset):
  def __init__(self):
    self.x_data = torch.randn(100, 4)                    # 100행 4열짜리 가상 특성 데이터
    self.y_data = torch.randint(0, 2, (100, 1)).float()  # 0이상 2미만 정수로 이루어진 100행 1열 정답

  def __len__(self):                                     # 전체 데이터셋의 총 샘플 개수 반환
    return len(self.x_data)
  def __getitem__(self, index):                           # 특정 인덱스의 데이터 한쌍(x, y)을 리턴
    return self.x_data[index], self.y_data[index]

dataset = MyCustomDataset()
print(f"데이터셋 총 샘플 수: {len(dataset)}")

# 첫번째 데이터 샘플 확인
sample_x, sample_y = dataset[0]
print(f"0번 샘플 데이터: {sample_x}")
print(f"0번 샘플 정답: {sample_y}")

- 배치: 마트에 카트(카트에 몇개까지 담는가?)
- 에포크: 데이터를 모두 훑으면 1에포크이다.

In [ ]:
dataloader = DataLoader(dataset, batch_size = 16, shuffle=True)
print(f"데이터로더의 길이: {len(dataloader)}개(100개 / 배치 크기 16)")

# 실제 학습 루프에서 DataLoader가 데이터를 꺼내는 모습
for epoch in range(2):
  print(f"\n -----[Epoch {epoch + 1}시작]")
  for batch_idx, (x_batch, y_batch) in enumerate(dataloader):
    # 첫번째 에포크의 7개 배치를 출력 확인
    if batch_idx < 7:
      print(f"Batch{batch_idx + 1} -> x 묶음 형태: {x_batch.shape}, y 묶음 형태: {y_batch.shape}")

단일 퍼셉트론

In [ ]:
import torch
import torch.nn as nn                         #PyTorch에서 인공 신경망을 구축하는데 필요한 핵심 모듈
import torch.optim as optim

torch.manual_seed(42)
x_data = torch.randn(100, 4)                  # 입력데이터
y_data = torch.randint(0, 2, (100, 1)).float() # 정답데이터
#간단한 신경망 모델 정
class SimpleNeuralNet(nn.Module):
  def __init__(self):
    super(SimpleNeuralNet, self).__init__()
    # 4개의 입력을 받아 1개의 출력(점수)을 만드는 선형레이어
    self.linear = nn.Linear(4,1)
    # 출력을 0과 1사이의 확률값으로 변환해주는 시그모이드 함
    self.sigmoid = nn.Sigmoid()
  def forward(self , x):
    out = self.linear(x)
    out = self.sigmoid(out)
    return out
# 모델, 오차함수 최적 알고리즘 설정
model = SimpleNeuralNet()                          # 객체 생성
criterion = nn.BCELoss()                           # 이진 분류용 교차 엔트로피 손실함수: sigmoid() 필요
optimizer = optim.SGD(model.parameters(), lr=0.01) # 최적화 도구 설정 : 모델의 파라미터를 전달해주고 경사하강법이 보폭(0.1)을 정한다.

epochs = 200
# 학습 루프
for epoch in range(1000):
  # 순전파: 모델에 데이터를 넣어 예측값 계산
  predictions = model (x_data)
  # 오차 계산
  loss = criterion(predictions, y_data)
  # 역전파: 미분값 계산
  optimizer.zero_grad()           # 이전 루프에서 계산된 미분값을 초기화
  loss.backward()                 # 역전파 수행
  # 가중치 업데이트
  optimizer.step()

# 20번 마다 학습 상태 출
  if(epoch + 1) % 20 == 0:
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")
print(" \n 학습완료")

다층 퍼셉트론

In [ ]:
import torch
import torch.nn as nn
# 1. nn.Module을 상속받는 고급 MLP 신경망 클래스 정의
class AdvanceMLP(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim): #사용할 레이어들을 정의한다.
    super(AdvanceMLP, self).__init__()

    self.fc1 = nn.Linear(input_dim, hidden_dim)    # 첫 번째 은닉층
    self.bn1 = nn.BatchNorm1d(hidden_dim)          # 배치 정규화 추가(학습 안정화)
    self.relu = nn.ReLU()                          # 비선형 활성화
    self.dropout = nn.Dropout(0.3)                 # 30% 뉴런 임의 차단(과적합 방지)

    self.fc2 = nn.Linear(hidden_dim, output_dim)   # 두 번째 은닉층
    self.segmoid = nn.Sigmoid()                    # 이진 분류용 시그모이드

  def forward(self, x):                            # 데이터가 신경망을 타고 흐르는 순서 지정
    x = self.fc1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.dropout(x)                            # 학습 시에만 작동하고 평가 시에는 자동으로 비활성화

    x = self.fc2(x)
    x = self.segmoid(x)
    return(x)


model = AdvanceMLP(input_dim=10, hidden_dim=32, output_dim =1)
print("직접 설게한 AdvanceMLP 신경망 구조")
print(model)

dummy_input = torch.randn(5, 10)
output = model(dummy_input)
print("입력 데이터 크기:", dummy_input.shape)
print("출력 데이터 크기:", output.shape)
print("출력 결과 형태(shape):", output)

1. PyTorch 신경망은 nn.Module을 상속받아 __init__에서 부품을 정의하고, forward에서 조립하는 방식으로 구현한다.
2. ReLU같은 비선형 활성화 함수가 있어야 신경망이 복잡한 패턴을 학습할 수 있다.
3. Dropout과 BatchNorm은 딥러닝의 고질적인 문제인 과적합을 방지하고 학습을 안정화하는 필수 방패막이다.